## This notebook is made to give the essence of RAG.

In [1]:
import os
from dotenv import load_dotenv
import json
from IPython.display import display,Markdown
from openai import OpenAI 
from litellm import completion
from pathlib import Path
import glob
import gradio as gr
load_dotenv(override = True)

c:\Users\Sushant\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
google_api_key = os.getenv("GOOGLE_API_KEY")


In [3]:
knowledge = {}
filenames = glob.glob("knowledge-base\employees\*")
print(filenames)
for filename in filenames:
    names = Path(filename).stem.split(" ")[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[names.lower()] = f.read()

<>:2: SyntaxWarning: invalid escape sequence '\e'
<>:2: SyntaxWarning: invalid escape sequence '\e'
C:\Users\Sushant\AppData\Local\Temp\ipykernel_17772\2142035036.py:2: SyntaxWarning: invalid escape sequence '\e'
  filenames = glob.glob("knowledge-base\employees\*")


['knowledge-base\\employees\\Alex Chen.md', 'knowledge-base\\employees\\Alex Harper.md', 'knowledge-base\\employees\\Alex Thomson.md', 'knowledge-base\\employees\\Amanda Foster.md', 'knowledge-base\\employees\\Avery Lancaster.md', 'knowledge-base\\employees\\Brandon Walker.md', 'knowledge-base\\employees\\Carlos Rodriguez.md', 'knowledge-base\\employees\\Daniel Park.md', 'knowledge-base\\employees\\David Kim.md', 'knowledge-base\\employees\\Emily Carter.md', 'knowledge-base\\employees\\Emily Tran.md', 'knowledge-base\\employees\\James Wilson.md', 'knowledge-base\\employees\\Jennifer Adams.md', 'knowledge-base\\employees\\Jessica Liu.md', 'knowledge-base\\employees\\Jordan Blake.md', 'knowledge-base\\employees\\Jordan K. Bishop.md', 'knowledge-base\\employees\\Kevin Zhang.md', 'knowledge-base\\employees\\Lisa Anderson.md', 'knowledge-base\\employees\\Marcus Johnson.md', 'knowledge-base\\employees\\Maxine Thompson.md', 'knowledge-base\\employees\\Maya Thompson.md', "knowledge-base\\emplo

In [ ]:
print(knowledge)


In [4]:
filenames = glob.glob("knowledge-base/products/*")
for filename in filenames:
    name = Path(filename).stem
    with open(filename,"r",encoding="utf-8") as p:
        knowledge[name.lower()] = p.read()

In [ ]:
print(knowledge)

In [ ]:
knowledge.keys()

In [5]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [6]:
def get_relevent_context(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_context =[]
    for word in words:
        for word in knowledge:
            relevant_context.append(knowledge[word])
    return relevant_context

## More enhanced way of doing the same stuff


In [7]:
def get_relevent_context(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

In [ ]:
get_relevent_context("who is lancaster?")

In [8]:
def additional_context(message):
    relevant = get_relevent_context(message)
    if not relevant:
        result = "This is no additional information to provide."
    else:
        result = "The following additional context might be relevant in answering the user's question: \n\n"
        result+= "\n\n".join(relevant)
    return result

In [ ]:
additional_context("who is lancaster and who is chen")

In [9]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = completion(model="gemini/gemini-2.5-flash",api_key=google_api_key,messages=messages)
    return response.choices[0].message.content

In [ ]:
view = gr.ChatInterface(chat).launch(inbrowser=True)

In [12]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=GEMINI_BASE_URL,api_key=google_api_key)

In [ ]:
history = []
user_history = []
while True:
    user_message = input("You: ")
    print(f"You: {user_message}")
    
    message = (
        [{"role": "system", "content": "SYSTEM_PREFIX"}]
        + history
        + [{"role": "user", "content": user_message}]
    )
    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=message,
        stream=True,
        api_key=google_api_key,
    )
    assistant_reply = ""
    print("Bot: ", end="") 

    for chunk in response:
        
        chunk_text = chunk.choices[0].delta.content or ""
        assistant_reply += chunk_text
        print(chunk_text, end="", flush=True)

    print()  

    
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": assistant_reply})

You: hi
Bot: Hi there! How can I help you today?
You: how are you
Bot: I don't have feelings or personal well-being like humans do, but I'm ready and functioning perfectly!

How are *you* doing today? And what can I help you with?
You: what can you do
Bot: That's a great question! As a large language model, I can do a lot of things with text and information. Here are some of my main capabilities:

1.  **Answer Questions:** I can provide information on a vast range of topics, from history and science to current events (up to my last training update) and pop culture.
2.  **Explain Concepts:** I can break down complex ideas into simpler terms, making them easier to understand.
3.  **Generate Creative Content:** I can write stories, poems, scripts, song lyrics, emails, letters, blog posts, marketing copy, and more, based on your prompts.
4.  **Summarize Information:** I can condense long articles, documents, or conversations into key points.
5.  **Translate Languages:** I can translate tex